In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM
from typing import Optional, Iterable, Dict, List
import os

import pandas as pd
import numpy as np
import warnings
import re
import matplotlib.pyplot as plt
import seaborn as sns
from cmcrameri import cm
import massbalancemachine as mbm
import logging
import torch.nn as nn
from skorch.helper import SliceDataset
from datetime import datetime
from skorch.callbacks import EarlyStopping, LRScheduler, Checkpoint
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset
import pickle 
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
from matplotlib.lines import Line2D

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"CROSS_REGION"
print(f"Base directory for data: {BASE_DIR}")

In [ ]:
MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc")
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

    vois_climate = [
        "t2m",
        "tp",
        "slhf",
        "sshf",
        "ssrd",
        "fal",
        "str",
    ]

vois_topographical = ["aspect", "slope", "svf"]

# Cross-regional modelling:

### Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"]

In [ ]:
rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

### Monthly datasets:

In [ ]:
# # Now test glaciers are all their glaciers:
# TEST_GLACIERS_SJM = load_stakes(cfg, "SJM").GLACIER.unique().tolist()
# TEST_GLACIERS_ISL = load_stakes(cfg, "ISL").GLACIER.unique().tolist()
# TEST_GLACIERS_NOR = load_stakes(cfg, "NOR").GLACIER.unique().tolist()
# TEST_GLACIERS_FR = load_stakes(cfg, "FR").GLACIER.unique().tolist()
# TEST_GLACIERS_IT_AT = load_stakes(cfg, "IT_AT").GLACIER.unique().tolist()
# TEST_GLACIERS_CH = load_stakes(cfg, "CH").GLACIER.unique().tolist()

# TEST_GLACIERS_BY_CODE = {
#     "SJM": TEST_GLACIERS_SJM,
#     "ISL": TEST_GLACIERS_ISL,
#     "NOR": TEST_GLACIERS_NOR,
#     "FR": TEST_GLACIERS_FR,
#     "IT_AT": TEST_GLACIERS_IT_AT,
#     "CH": TEST_GLACIERS_CH
# }


In [ ]:
SOURCE_REGIONS = ['CH', 'NOR', 'ISL', 'CEU']
experiment_region_groups = {"CEU": ["FR", "IT_AT", "CH"]}
res_xreg_by_source = {}
split_info_by_source = {}

for src_region in SOURCE_REGIONS:
    print(f"\n{'='*50}")
    print(f"Source region: {src_region}")

    res_xreg, split_info = prepare_monthly_df_xreg_SOURCE_to_EU(
        cfg=cfg,
        dfs=dfs,
        paths=paths,
        vois_climate=vois_climate,
        vois_topographical=vois_topographical,
        run_flag=False,
        source_code=src_region,
        region_groups=experiment_region_groups)

    res_xreg_by_source[src_region] = res_xreg
    split_info_by_source[src_region] = split_info

    df_train = res_xreg["df_train"]
    df_test = res_xreg["df_test"]

    print(f"Train glaciers ({src_region}):", len(split_info["train_glaciers"]))
    print(f"Test glaciers (non-{src_region}):",
          len(split_info["test_glaciers"]))
    print("Train rows:", len(df_train), "Test rows:", len(df_test))

### Domains shifts & feature overlap:

#### Domain shift:

In [ ]:
res_all_xreg_by_source = {}

# Maps each source to the codes it "owns" (used to exclude from targets)
SOURCE_MEMBERS = {
    "CH": ["CH"],
    "NOR": ["NOR"],
    "ISL": ["ISL"],
    "CEU": ["FR", "IT_AT", "CH"],
}

for src_region in SOURCE_REGIONS:
    src_members = SOURCE_MEMBERS[src_region]

    # Build CEU group only if none of its members are the source
    ceu_members_excl_src = [
        c for c in ["FR", "IT_AT", "CH"] if c not in src_members
    ]
    experiment_region_groups = {}
    if ceu_members_excl_src:
        experiment_region_groups["CEU"] = ceu_members_excl_src

    res_all_xreg = build_xreg_res_all(
        res_xreg=res_xreg_by_source[src_region],
        target_source_codes=None,
        source_col="SOURCE_CODE",
        key_prefix=f"XREG_{src_region}_TO",
        ch_code=src_region,  # still used to exclude individual entries
        region_groups=experiment_region_groups,
    )

    res_all_xreg_by_source[src_region] = res_all_xreg
    print(f"{src_region} -> keys:", list(res_all_xreg.keys()))

In [ ]:
CLIMATE_COLS = ['t2m', 'tp', 'slhf', 'sshf', 'ssrd', 'fal', 'str']
TOPO_COLS = ['aspect', 'slope', 'svf', 'ELEVATION_DIFFERENCE']

SHIFTS_CACHE = BASE_DIR / "domain_shifts_cache.pkl"

if SHIFTS_CACHE.exists():
    with open(SHIFTS_CACHE, "rb") as f:
        cache = pickle.load(f)
    scaler_m = cache["scaler_m"]
    scaler_s = cache["scaler_s"]
    blur_m = cache["blur_m"]
    blur_s = cache["blur_s"]
    bandwidths_m = cache["bandwidths_m"]
    bandwidths_s = cache["bandwidths_s"]
    all_shifts_by_source = cache["all_shifts_by_source"]
    print(f"Loaded shifts from cache: {SHIFTS_CACHE}")

else:
    scaler_m, scaler_s = build_global_scalers_multi_source(
        res_xreg_by_source,
        monthly_cols=CLIMATE_COLS,
        static_cols=TOPO_COLS,
    )

    blur_m, blur_s = estimate_global_bandwidths(
        res_xreg_by_source,
        monthly_cols=CLIMATE_COLS,
        static_cols=TOPO_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        seed=cfg.seed,
    )
    bandwidths_m = [blur_m * 0.5, blur_m, blur_m * 2.0]
    bandwidths_s = [blur_s * 0.5, blur_s, blur_s * 2.0]

    all_shifts_by_source = {}
    for src_region in SOURCE_REGIONS:
        all_shifts = {}
        for key in tqdm(res_all_xreg_by_source[src_region], desc=src_region):
            shift = compute_domain_shift(
                df_src=res_all_xreg_by_source[src_region][key]["df_train"],
                df_tgt=res_all_xreg_by_source[src_region][key]["df_test"],
                monthly_cols=CLIMATE_COLS,
                static_cols=TOPO_COLS,
                scaler_m=scaler_m,
                scaler_s=scaler_s,
                blur_m=blur_m,
                blur_s=blur_s,
                bandwidths_m=bandwidths_m,
                bandwidths_s=bandwidths_s,
                compute_marginals=False,
                seed=cfg.seed,
            )
            all_shifts[key] = shift
        all_shifts_by_source[src_region] = all_shifts

    with open(SHIFTS_CACHE, "wb") as f:
        pickle.dump(
            {
                "scaler_m": scaler_m,
                "scaler_s": scaler_s,
                "blur_m": blur_m,
                "blur_s": blur_s,
                "bandwidths_m": bandwidths_m,
                "bandwidths_s": bandwidths_s,
                "all_shifts_by_source": all_shifts_by_source,
            }, f)
    print(f"Saved shifts to cache: {SHIFTS_CACHE}")

In [ ]:
# # Plot per-pair shifts
# for src_region, all_shifts in all_shifts_by_source.items():
#     for key, shift in all_shifts.items():
#         tgt = key.split('_TO_')[1]
#         print(f"Shift for {key}:")
#         fig = plot_domain_shift(shift,
#                                 CLIMATE_COLS,
#                                 TOPO_COLS,
#                                 src=src_region,
#                                 tgt=tgt)
#         plt.show()

# Plot summary across regions, one figure per source
for src_region, all_shifts in all_shifts_by_source.items():
    print(f"\nDomain shift summary for source: {src_region}")
    fig = plot_domain_shift_across_regions(all_shifts, src_region=src_region)
    plt.show()

In [ ]:
for key in res_all_xreg:
    Num_measurements = res_all_xreg[key]['df_test']['ID'].nunique()
    print(key, 'Target region num_measurements:', Num_measurements)

#### KDEs:

In [ ]:
# scaler_m_pmb, scaler_s_pmb = build_global_scalers_multi_source(
#     res_xreg_by_source,
#     monthly_cols=MONTHLY_COLS + ["POINT_BALANCE"],
#     static_cols=STATIC_COLS,
# )

# # for src_region in SOURCE_REGIONS:
# #     figs = plot_feature_kde_overlap_xreg_with_shifts(
# #         res_all_xreg=res_all_xreg_by_source[src_region],
# #         cfg=cfg,
# #         monthly_cols=MONTHLY_COLS,
# #         static_cols=STATIC_COLS,
# #         scaler_m=scaler_m_pmb,
# #         scaler_s=scaler_s_pmb,
# #         output_dir=f"figures/xreg_kde_with_shifts/{src_region}",
# #     )

# figs = plot_feature_kde_overlap_xreg_with_shifts(
#     res_all_xreg=res_all_xreg_by_source["CH"],
#     cfg=cfg,
#     monthly_cols=MONTHLY_COLS,
#     static_cols=STATIC_COLS,
#     scaler_m=scaler_m_pmb,
#     scaler_s=scaler_s_pmb,
#     output_dir=f"figures/xreg_kde_with_shifts/{src_region}",
# )

#### T-sne:

In [ ]:
# for src_region in SOURCE_REGIONS:
#     figs_by_code = plot_tsne_overlap_xreg_from_single_res(
#         res_xreg=res_xreg_by_source[src_region],
#         cfg=cfg,
#         STATIC_COLS=STATIC_COLS,
#         MONTHLY_COLS=MONTHLY_COLS,
#         group_col="SOURCE_CODE",
#         target_code=src_region,
#         use_aug=False,
#         n_iter=1000,
#     )

## LSTM model
### LSTM datasets:

In [ ]:
logs_cache_dir = BASE_DIR / "LSTM_cache"
outputs_xreg_by_source = {}

for src_region in SOURCE_REGIONS:
    if src_region != "CH":
        experiment_region_groups = {
            "CEU": ["FR", "IT_AT", "CH"],
            "USCA": ["ALA", "CAW"],
        }
    else:
        experiment_region_groups = {
            "CEU": ["FR", "IT_AT"],
            "USCA": ["ALA", "CAW"],
        }

    outputs_xreg_by_source[src_region] = build_or_load_lstm_all_xreg(
        cfg=cfg,
        res_xreg=res_xreg_by_source[src_region],
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=logs_cache_dir / src_region,
        ch_code=src_region,
        region_groups=experiment_region_groups,
    )
    print(f"{src_region} -> LSTM dataset keys:",
          list(outputs_xreg_by_source[src_region].keys()))

### LSTM parameters:

In [ ]:
default_params = {
    'Fm': 8,
    'Fs': 3,
    'hidden_size': 128,
    'num_layers': 1,
    'bidirectional': False,
    'dropout': 0.0,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0.1,
    'lr': 0.001,
    'weight_decay': 1e-05,
    'loss_name': 'neutral',
    'two_heads': False,
    'head_dropout': 0.1,
    'loss_spec': None
}

### Train model:

In [ ]:
def train_or_load_one_source_model(
    cfg,
    key: str,
    lstm_assets: dict,  # only needs ds_train, train_idx, val_idx
    best_params: dict,
    device,
    models_dir="models",
    prefix="lstm_xreg",
    train_flag=True,
    force_retrain=False,
    epochs=150,
    batch_size_train=64,
    batch_size_val=128,
    verbose=True,
    date=None,
):
    """Train or load a single source-region model. No test set needed."""
    run_date = datetime.now().strftime("%Y-%m-%d") if date is None else date
    os.makedirs(models_dir, exist_ok=True)
    model_filename = os.path.join(models_dir, f"{prefix}_{key}_{run_date}.pt")

    model = mbm.models.LSTM_MB.build_model_from_params(cfg,
                                                       best_params,
                                                       device,
                                                       verbose=verbose)
    loss_fn = mbm.models.LSTM_MB.resolve_loss_fn(best_params)

    # Load existing checkpoint if available and not forcing retrain
    if train_flag and (not force_retrain) and os.path.exists(model_filename):
        state = torch.load(model_filename, map_location=device)
        model.load_state_dict(state)
        return model, model_filename, None

    if not train_flag and not os.path.exists(model_filename):
        raise FileNotFoundError(
            f"No checkpoint found for {key}: {model_filename}")

    if not train_flag and os.path.exists(model_filename):
        state = torch.load(model_filename, map_location=device)
        model.load_state_dict(state)
        return model, model_filename, None

    # --- Train ---
    mbm.utils.seed_all(cfg.seed)

    ds_train = lstm_assets["ds_train"]
    train_idx = lstm_assets["train_idx"]
    val_idx = lstm_assets["val_idx"]

    ds_train_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_train)

    train_dl, val_dl = ds_train_copy.make_loaders(
        train_idx=train_idx,
        val_idx=val_idx,
        batch_size_train=batch_size_train,
        batch_size_val=batch_size_val,
        seed=cfg.seed,
        fit_and_transform=True,
        shuffle_train=True,
        use_weighted_sampler=True,
        verbose=verbose,
    )

    if os.path.exists(model_filename):
        os.remove(model_filename)
        if verbose:
            print(f"Deleted existing model file: {model_filename}")

    history, best_val, best_state = model.train_loop(
        device=device,
        train_dl=train_dl,
        val_dl=val_dl,
        epochs=epochs,
        lr=best_params["lr"],
        weight_decay=best_params["weight_decay"],
        clip_val=1,
        sched_factor=0.5,
        sched_patience=6,
        sched_threshold=0.01,
        sched_threshold_mode="rel",
        sched_cooldown=1,
        sched_min_lr=1e-6,
        es_patience=15,
        es_min_delta=1e-4,
        log_every=5,
        verbose=verbose,
        save_best_path=model_filename,
        loss_fn=loss_fn,
    )

    if verbose:
        plot_history_lstm(history)

    state = torch.load(model_filename, map_location=device)
    model.load_state_dict(state)

    return model, model_filename, {"history": history, "best_val": best_val}


def prepare_test_loader_for_target(
    cfg,
    model,
    lstm_assets_src: dict,  # needs ds_train (for scaler fitting)
    lstm_assets_tgt: dict,  # needs ds_test
    batch_size_test=128,
):
    """Given a trained model and a target's test assets, build the test loader."""
    ds_train_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        lstm_assets_src["ds_train"])
    ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        lstm_assets_tgt["ds_test"])

    # fit scaler on train, apply to test
    ds_train_copy.make_loaders(
        train_idx=lstm_assets_src["train_idx"],
        val_idx=lstm_assets_src["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
    )

    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_test_copy, ds_train_copy, batch_size=batch_size_test, seed=cfg.seed)

    return test_dl, ds_test_copy

In [ ]:
models_by_source = {}
infos_by_source = {}

for src_region in SOURCE_REGIONS:
    # Pick any key to get the train assets — they're the same across all targets
    any_key = next(iter(outputs_xreg_by_source[src_region]))
    train_assets = outputs_xreg_by_source[src_region][any_key]

    model, model_path, info = train_or_load_one_source_model(
        cfg=cfg,
        key=src_region,
        lstm_assets=train_assets,
        best_params=default_params,
        device=device,
        models_dir=BASE_DIR / "models" / src_region,
        prefix=f"lstm_xreg_{src_region}",
        force_retrain=False,
        epochs=150,
        date='2026-04-09')

    models_by_source[src_region] = model
    infos_by_source[src_region] = {"model_path": model_path, **(info or {})}
    print(f"{src_region} -> model trained/loaded from {model_path}")

### Evaluate on test:

In [ ]:
df_metrics_by_source = {}
preds_by_source = {}
figs_by_source = {}

for src_region in SOURCE_REGIONS:
    print(f"\nEvaluating source region: {src_region}")
    # One model per source, broadcast to all target keys
    target_keys = outputs_xreg_by_source[src_region].keys()
    models_by_key = {key: models_by_source[src_region] for key in target_keys}

    if src_region == "CH":
        custom_order = ["CEU", "NOR", "ISL", "SJM", "USCA"]
    elif src_region == "NOR":
        custom_order = ["CEU", "ISL", "SJM", "USCA"]
    elif src_region == "ISL":
        custom_order = ["CEU", "NOR", "SJM", "USCA"]

    df_metrics, preds_by_key, figs_by_key, fig_grid = evaluate_all_models(
        cfg=cfg,
        models_by_key=models_by_key,
        lstm_assets_by_key=outputs_xreg_by_source[src_region],
        device=device,
        save_dir=f"figures/eval_xreg/{src_region}",
        ax_xlim=(-16, 10),
        ax_ylim=(-16, 10),
        grid_shape=(2, 3),
        grid_figsize=(25, 12),
        domain_shifts=all_shifts_by_source[src_region],
        complement_key=f'XREG_{src_region}_TO_',
        custom_order=custom_order,
    )

    df_metrics_by_source[src_region] = df_metrics
    preds_by_source[src_region] = preds_by_key
    figs_by_source[src_region] = figs_by_key

### Region shift vs performance:

In [ ]:
def plot_region_shift_vs_performance(
    df_metrics: pd.DataFrame,
    all_shifts: dict,
    complement_key: str = "",
    performance_cols: list[str] | None = None,
    distance_cols: list[str] | None = None,
):
    from scipy import stats

    if performance_cols is None:
        performance_cols = [
            c for c in df_metrics.columns
            if any(kw in c.lower() for kw in ["rmse", "bias"])
        ]
        if not performance_cols:
            raise ValueError(
                f"No performance columns auto-detected in df_metrics.\n"
                f"Available columns: {list(df_metrics.columns)}\n"
                f"Pass performance_cols= explicitly.")
        print(f"Auto-detected performance columns: {performance_cols}")

    if distance_cols is None:
        distance_cols = [
            "D_mmd2_joint",
            "D_mmd2_climate",
            "D_mmd2_topo",
            "D_energy_joint",
            "D_sinkhorn_joint",
            "D_sinkhorn_climate",
            "D_sinkhorn_topo",
        ]

    # --- build flat region DataFrame ---
    records = []
    for full_key in df_metrics.index:
        shift_key = f"{complement_key}{full_key}" if complement_key else full_key
        if shift_key not in all_shifts:
            print(f"Warning: '{shift_key}' not in all_shifts, skipping.")
            continue

        shift = all_shifts[shift_key]

        parts = shift_key.split("_TO_")
        src = parts[0].replace("XREG_", "")
        tgt = parts[1] if len(parts) > 1 else shift_key
        region_label = f"{src}→{tgt}"

        row = {
            "full_key": full_key,
            "region": region_label,
            "src": src,
            "tgt": tgt
        }
        for pc in performance_cols:
            row[pc] = df_metrics.loc[full_key, pc]
        for dc in distance_cols:
            row[dc] = shift.get(dc, float("nan"))
        records.append(row)

    if not records:
        raise ValueError(
            "No matching regions between df_metrics and all_shifts.\n"
            f"df_metrics index: {list(df_metrics.index)}\n"
            f"all_shifts keys:  {list(all_shifts.keys())}")

    df_region = pd.DataFrame(records)
    n_regions = len(df_region)
    print(
        f"Plotting {n_regions} source→target pairs: {list(df_region['region'])}"
    )

    # --- colours: one colour per source region, marker per target ---
    unique_sources = df_region["src"].unique()
    src_colors = {
        s: c
        for s, c in zip(unique_sources,
                        get_cmap_hex(cm.batlow, max(len(unique_sources), 4)))
    }

    xlabel_map = {
        "D_mmd2_joint": "MMD² joint",
        "D_mmd2_climate": "MMD² climate",
        "D_mmd2_topo": "MMD² topo",
        "D_energy_joint": "Energy distance joint",
        "D_energy_climate": "Energy distance climate",
        "D_energy_topo": "Energy distance topo",
        "D_sinkhorn_joint": "Sinkhorn distance joint",
        "D_sinkhorn_climate": "Sinkhorn distance climate",
        "D_sinkhorn_topo": "Sinkhorn distance topo",
    }

    nrows = len(performance_cols)
    ncols = len(distance_cols)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4.5 * ncols, 4.2 * nrows),
        squeeze=False,
    )

    for r, pc in enumerate(performance_cols):
        for c, dc in enumerate(distance_cols):
            ax = axes[r][c]

            x = df_region[dc].values.astype(float)
            y = df_region[pc].values.astype(float)
            mask = np.isfinite(x) & np.isfinite(y)

            for _, row in df_region.iterrows():
                xi = float(row[dc])
                yi = float(row[pc])
                if not (np.isfinite(xi) and np.isfinite(yi)):
                    continue
                ax.scatter(
                    xi,
                    yi,
                    color=src_colors[row["src"]],
                    s=140,
                    zorder=3,
                    edgecolors="white",
                    linewidths=0.6,
                )
                ax.annotate(
                    row["region"],
                    (xi, yi),
                    xytext=(6, 4),
                    textcoords="offset points",
                    fontsize=9,
                    fontweight="bold",
                    color=src_colors[row["src"]],
                )

            # --- Spearman ---
            n_valid = mask.sum()
            if n_valid >= 3:
                rho, pval = stats.spearmanr(x[mask], y[mask])
                corr_txt = f"rho = {rho:.2f}  p = {pval:.2f}  n = {n_valid}"
            else:
                corr_txt = f"n = {n_valid} (too few for rho)"

            ax.text(
                0.04,
                0.97,
                corr_txt,
                transform=ax.transAxes,
                va="top",
                ha="left",
                fontsize=8,
                bbox=dict(boxstyle="round,pad=0.3",
                          fc="white",
                          ec="none",
                          alpha=0.8),
            )

            ax.set_xlabel(xlabel_map.get(dc, dc), fontsize=9)
            ax.set_ylabel(pc, fontsize=9)
            ax.grid(color="#e0e0e0", linewidth=0.5)
            ax.spines[["top", "right"]].set_visible(False)
            ax.set_axisbelow(True)

    # --- legend: one entry per source ---
    legend_handles = [
        plt.scatter([], [], color=src_colors[s], s=80, label=s)
        for s in unique_sources
    ]
    fig.legend(
        handles=legend_handles,
        title="Source region",
        loc="upper right",
        bbox_to_anchor=(1.02, 1),
        frameon=False,
    )

    fig.suptitle(
        "Region-level domain shift vs transfer performance\n(all source regions, zero-shot)",
        fontsize=13,
        y=1.01,
    )
    plt.tight_layout()
    return fig, df_region

In [ ]:
EXCLUDE_TARGETS = ["USCA"]  # set to [] to include all
# EXCLUDE_TARGETS = []

# Combine across all sources
df_metrics_all = pd.concat(df_metrics_by_source.values())
all_shifts_all = {}
for src_region in SOURCE_REGIONS:
    all_shifts_all.update(all_shifts_by_source[src_region])

# Filter out excluded targets
if EXCLUDE_TARGETS:
    mask = ~df_metrics_all.index.str.contains("|".join(EXCLUDE_TARGETS))
    df_metrics_all = df_metrics_all[mask]
    all_shifts_all = {
        k: v
        for k, v in all_shifts_all.items()
        if not any(tgt in k for tgt in EXCLUDE_TARGETS)
    }

In [ ]:
fig, _ = plot_region_shift_vs_performance(df_metrics_all, all_shifts_all)

In [ ]:
def plot_region_shift_vs_performance_single_d(
        df_metrics: pd.DataFrame,
        all_shifts: dict,
        complement_key: str = "",
        performance_cols: list[str] | None = None,
        distance_variant: str = "mmd2",  # "mmd2", "energy", or "sinkhorn"
):
    from scipy import stats

    if performance_cols is None:
        performance_cols = [
            c for c in df_metrics.columns
            if any(kw in c.lower() for kw in ["rmse", "bias"])
        ]
        if not performance_cols:
            raise ValueError(
                f"No performance columns auto-detected in df_metrics.\n"
                f"Available columns: {list(df_metrics.columns)}\n"
                f"Pass performance_cols= explicitly.")
        print(f"Auto-detected performance columns: {performance_cols}")

    distance_cols = [
        f"D_{distance_variant}_joint",
        f"D_{distance_variant}_climate",
        f"D_{distance_variant}_topo",
    ]

    xlabel_map = {
        f"D_{distance_variant}_joint": f"{distance_variant} joint",
        f"D_{distance_variant}_climate": f"{distance_variant} climate",
        f"D_{distance_variant}_topo": f"{distance_variant} topo",
    }

    # --- build flat region DataFrame ---
    records = []
    for full_key in df_metrics.index:
        shift_key = f"{complement_key}{full_key}" if complement_key else full_key
        if shift_key not in all_shifts:
            print(f"Warning: '{shift_key}' not in all_shifts, skipping.")
            continue

        shift = all_shifts[shift_key]

        parts = shift_key.split("_TO_")
        src = parts[0].replace("XREG_", "")
        tgt = parts[1] if len(parts) > 1 else shift_key
        region_label = f"{src}→{tgt}"

        row = {
            "full_key": full_key,
            "region": region_label,
            "src": src,
            "tgt": tgt,
        }
        for pc in performance_cols:
            row[pc] = df_metrics.loc[full_key, pc]
        for dc in distance_cols:
            row[dc] = shift.get(dc, float("nan"))
        records.append(row)

    if not records:
        raise ValueError(
            "No matching regions between df_metrics and all_shifts.\n"
            f"df_metrics index: {list(df_metrics.index)}\n"
            f"all_shifts keys:  {list(all_shifts.keys())}")

    df_region = pd.DataFrame(records)
    print(
        f"Plotting {len(df_region)} source→target pairs: {list(df_region['region'])}"
    )

    unique_sources = df_region["src"].unique()
    src_colors = {
        s: c
        for s, c in zip(unique_sources,
                        get_cmap_hex(cm.batlow, max(len(unique_sources), 4)))
    }

    nrows = len(performance_cols)
    ncols = 3  # joint, climate, topo

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4.5 * ncols, 4.2 * nrows),
        squeeze=False,
    )

    for r, pc in enumerate(performance_cols):
        for c, dc in enumerate(distance_cols):
            ax = axes[r][c]

            x = df_region[dc].values.astype(float)
            y = df_region[pc].values.astype(float)
            mask = np.isfinite(x) & np.isfinite(y)

            for _, row in df_region.iterrows():
                xi = float(row[dc])
                yi = float(row[pc])
                if not (np.isfinite(xi) and np.isfinite(yi)):
                    continue
                ax.scatter(
                    xi,
                    yi,
                    color=src_colors[row["src"]],
                    s=140,
                    zorder=3,
                    edgecolors="white",
                    linewidths=0.6,
                )
                ax.annotate(
                    row["region"],
                    (xi, yi),
                    xytext=(6, 4),
                    textcoords="offset points",
                    fontsize=9,
                    fontweight="bold",
                    color=src_colors[row["src"]],
                )

            n_valid = mask.sum()
            if n_valid >= 3:
                rho, pval = stats.spearmanr(x[mask], y[mask])
                corr_txt = f"rho = {rho:.2f}  p = {pval:.2f}  n = {n_valid}"
            else:
                corr_txt = f"n = {n_valid} (too few for rho)"

            ax.text(
                0.04,
                0.97,
                corr_txt,
                transform=ax.transAxes,
                va="top",
                ha="left",
                fontsize=8,
                bbox=dict(boxstyle="round,pad=0.3",
                          fc="white",
                          ec="none",
                          alpha=0.8),
            )

            ax.set_xlabel(xlabel_map[dc], fontsize=9)
            ax.set_ylabel(pc, fontsize=9)
            ax.grid(color="#e0e0e0", linewidth=0.5)
            ax.spines[["top", "right"]].set_visible(False)
            ax.set_axisbelow(True)

    legend_handles = [
        plt.scatter([], [], color=src_colors[s], s=80, label=s)
        for s in unique_sources
    ]
    fig.legend(
        handles=legend_handles,
        title="Source region",
        loc="upper right",
        bbox_to_anchor=(1.02, 1),
        frameon=False,
    )

    fig.suptitle(
        f"Region-level domain shift vs transfer performance — {distance_variant}",
        fontsize=13,
        y=1.01,
    )
    plt.tight_layout()
    return fig, df_region

In [ ]:
fig, _ = plot_region_shift_vs_performance_single_d(
    df_metrics_all,
    all_shifts_all,
    distance_variant="mmd2",
    performance_cols=["RMSE_annual", "RMSE_winter"])

In [ ]:
fig, _ = plot_region_shift_vs_performance_single_d(
    df_metrics_all,
    all_shifts_all,
    distance_variant="energy",
    performance_cols=["RMSE_annual", "RMSE_winter"])

In [ ]:
fig, _ = plot_region_shift_vs_performance_single_d(
    df_metrics_all,
    all_shifts_all,
    distance_variant="sinkhorn",
    performance_cols=["RMSE_annual", "RMSE_winter"])

### Glacier granularity:

In [ ]:
def compute_glacier_performance(
    preds: pd.DataFrame,
    glacier_col: str = "GLACIER",
    min_ids: int = 3,
) -> pd.DataFrame:
    """
    Aggregate predictions to glacier level.
    Returns RMSE and bias per glacier per period.
    """
    records = []
    for (glacier, period), df_gl in preds.groupby([glacier_col, "PERIOD"]):
        if df_gl["ID"].nunique() < min_ids:
            continue
        rmse = np.sqrt(mean_squared_error(df_gl["target"], df_gl["pred"]))
        bias = (df_gl["pred"] - df_gl["target"]).mean()
        records.append({
            glacier_col: glacier,
            "PERIOD": period,
            "RMSE": rmse,
            "Bias": bias,
            "n_ids": df_gl["ID"].nunique(),
        })

    return pd.DataFrame(records).set_index(glacier_col)

In [ ]:
MIN_IDS_GLACIER = 3  # minimum unique measurements per glacier

# --- Run for all source→target combinations ---
glacier_shifts_by_source = {}
glacier_perf_by_source = {}

for src_region in SOURCE_REGIONS:
    df_train = res_xreg_by_source[src_region]["df_train"]
    glacier_shifts = {}
    glacier_perf = {}

    for key, res in res_all_xreg_by_source[src_region].items():
        tgt = key.split("_TO_")[1]
        if any(excl in tgt for excl in EXCLUDE_TARGETS):
            continue

        print(f"Computing glacier-level shift: {key}")
        df_test = res["df_test"]
        preds = preds_by_source[src_region][key]

        glacier_shifts[key] = compute_glacier_shift_vs_source(
            df_train=df_train,
            df_test=df_test,
            monthly_cols=CLIMATE_COLS,
            static_cols=TOPO_COLS,
            scaler_m=scaler_m,
            scaler_s=scaler_s,
            min_ids=MIN_IDS_GLACIER,
        )

        glacier_perf[key] = compute_glacier_performance(
            preds,
            min_ids=MIN_IDS_GLACIER,
        )

    glacier_shifts_by_source[src_region] = glacier_shifts
    glacier_perf_by_source[src_region] = glacier_perf

In [ ]:
def plot_glacier_shift_vs_performance(
    glacier_shifts_by_source: dict,
    glacier_perf_by_source: dict,
    performance_col: str = "RMSE",
    period: str = "annual",
    distance_cols: list[str] = [
        "D_sinkhorn_joint", "D_sinkhorn_climate", "D_sinkhorn_topo"
    ],
):
    from scipy import stats

    records = []
    for src_region, shifts_by_key in glacier_shifts_by_source.items():
        for key, df_shift in shifts_by_key.items():
            tgt = key.split("_TO_")[1]
            df_perf = glacier_perf_by_source[src_region][key]
            df_perf = df_perf[df_perf["PERIOD"] == period]

            df_joined = df_shift.join(df_perf[[performance_col, "n_ids"]],
                                      how="inner",
                                      rsuffix="_perf")
            for glacier, row in df_joined.iterrows():
                rec = {
                    "glacier": glacier,
                    "src": src_region,
                    "tgt": tgt,
                    "label": f"{src_region}→{tgt}",
                    performance_col: row[performance_col],
                    "n_ids": row["n_ids"],
                }
                for dc in distance_cols:
                    rec[dc] = row[dc]
                records.append(rec)

    df = pd.DataFrame(records)
    print(f"Total glaciers: {len(df)}")

    unique_sources = df["src"].unique()
    src_colors = {
        s: c
        for s, c in zip(unique_sources,
                        get_cmap_hex(cm.batlow, max(len(unique_sources), 4)))
    }

    xlabel_map = {
        "D_sinkhorn_joint": "Sinkhorn joint",
        "D_sinkhorn_climate": "Sinkhorn climate",
        "D_sinkhorn_topo": "Sinkhorn topo",
    }

    ncols = len(distance_cols)
    fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 5), squeeze=False)

    for c, dc in enumerate(distance_cols):
        ax = axes[0][c]

        for src, grp in df.groupby("src"):
            x = grp[dc].values.astype(float)
            y = grp[performance_col].values.astype(float)
            mask = np.isfinite(x) & np.isfinite(y)
            ax.scatter(
                x[mask],
                y[mask],
                color=src_colors[src],
                s=8,  # small fixed size
                alpha=0.5,
                label=src,
                edgecolors="none")

        # Overall Spearman
        x_all = df[dc].values.astype(float)
        y_all = df[performance_col].values.astype(float)
        mask_all = np.isfinite(x_all) & np.isfinite(y_all)
        rho, pval = stats.spearmanr(x_all[mask_all], y_all[mask_all])
        ax.text(0.04,
                0.97,
                f"rho = {rho:.2f}  p = {pval:.3f}  n = {mask_all.sum()}",
                transform=ax.transAxes,
                va="top",
                fontsize=9,
                bbox=dict(boxstyle="round,pad=0.3",
                          fc="white",
                          ec="none",
                          alpha=0.8))

        ax.set_xlabel(xlabel_map.get(dc, dc), fontsize=10)
        ax.set_ylabel(f"{performance_col} ({period})",
                      fontsize=10) if c == 0 else None
        ax.grid(color="#e0e0e0", linewidth=0.5)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_axisbelow(True)

        if c == ncols - 1:
            ax.legend(title="Source", frameon=False, fontsize=8)

    fig.suptitle("Glacier-level domain shift vs transfer performance",
                 fontsize=12)
    plt.tight_layout()
    return fig, df


fig, df_glacier = plot_glacier_shift_vs_performance(
    glacier_shifts_by_source,
    glacier_perf_by_source,
    performance_col="RMSE",
    period="annual",
)
plt.show()